In [26]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ReadS3Data") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.DefaultAWSCredentialsProviderChain"
    ) \
    .getOrCreate()

In [27]:
from pyspark.sql.functions import sum, col

def check_null(df):
    null_count = df.select([sum(col(c).isNull().cast('int').alias(c)) for c in df.columns])

    null_count.show()

In [28]:
df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('quote', '"') \
    .option('escape', '"') \
    .option('multiline', 'true') \
    .csv('s3a://de-project-raw-abhi/raw/sales.csv')

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   C

In [29]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)



Cleaning column names

In [30]:
import re

def clean_columns(df):
    new_cols = []

    for c in df.columns:
        c = c.strip()
        c = c.lower()
        c = re.sub(r"[^\w]+", "_", c)       # replace special characters with _
        c = re.sub(r"_+", "_", c)           # remove multiple underscores
        c = c.strip('_')
        new_cols.append(c)

    return df.toDF(*new_cols)


df_column_remnamed = clean_columns(df)

In [31]:
df_column_remnamed.printSchema()

root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)



In [32]:
df_column_remnamed.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|    segment|      country|           city|         state|postal_code| region|     product_id|       category|sub_category|        product_name|   sales|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|0

In [33]:
check_null(df_column_remnamed)

+--------------------------------------------+------------------------------------------------+----------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------------------------------------------------------+----------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------------------------+------------------------------------------+------------------------------------------------------+--------------------------------------------+----------------------------------------------------+------------------------------------------------+--------------------------------------------------------+--------------------------------------------------------+------------------------------------------+
|sum(CAST((row_id IS NULL) AS INT) AS row_id)|sum(CAST((order_id IS NULL) AS INT) AS order_id)|sum

Changing datatypes of the column

In [36]:
from pyspark.sql.functions import col, to_date

df_clean = df_column_remnamed.withColumn('order_date', to_date(col('order_date'), 'dd/MM/yyyy')) \
    .withColumn('ship_date', to_date(col('ship_date'), 'dd/MM/yyyy')) \
    .withColumn('postal_code', col('postal_code').cast('string'))

In [37]:
df_clean.printSchema()

root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)



In [38]:
df_clean.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|    segment|      country|           city|         state|postal_code| region|     product_id|       category|sub_category|        product_name|   sales|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|2017-11-08|2017-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|2

Checking null values

In [39]:
check_null(df_clean)

+--------------------------------------------+------------------------------------------------+----------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------------------------------------------------------+----------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------------------------+------------------------------------------+------------------------------------------------------+--------------------------------------------+----------------------------------------------------+------------------------------------------------+--------------------------------------------------------+--------------------------------------------------------+------------------------------------------+
|sum(CAST((row_id IS NULL) AS INT) AS row_id)|sum(CAST((order_id IS NULL) AS INT) AS order_id)|sum

In [41]:
df_clean.filter(col('postal_code').isNull()).show()

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+----------+-------+-----------+------+---------------+---------------+------------+--------------------+-------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|   customer_name|    segment|      country|      city|  state|postal_code|region|     product_id|       category|sub_category|        product_name|  sales|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+----------+-------+-----------+------+---------------+---------------+------------+--------------------+-------+
|  2235|CA-2018-104066|2018-12-05|2018-12-10|Standard Class|   QJ-19255|    Quincy Jones|  Corporate|United States|Burlington|Vermont|       NULL|  East|TEC-AC-10001013|     Technology| Accessories|Logitech ClearCha...| 205.03|
|  5275|CA-2016-162887|2016-11-07|2016-11-09|  Second Class|   SV-20785|Stewart Visinsky

In [44]:
df_clean.distinct().count()

9800

In [45]:
df_clean.count()

9800

In [48]:
df_clean.groupBy('region').count().show()

+-------+-----+
| region|count|
+-------+-----+
|  South| 1598|
|Central| 2277|
|   East| 2785|
|   West| 3140|
+-------+-----+



In [ ]:
df_clean.write \
    .mode('overwrite') \
    .partitionBy('region') \
    .parquet('s3a://de-project-staging-abhi/sales/')

In [55]:
df_silver = spark.read.parquet("s3a://de-project-staging-abhi/sales/")
df_silver.show()

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+---------------+---------------+------------+--------------------+--------+------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|  segment|      country|         city|     state|postal_code|     product_id|       category|sub_category|        product_name|   sales|region|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+---------------+---------------+------------+--------------------+--------+------+
|     3|CA-2017-138688|2017-06-12|2017-06-16|  Second Class|   DV-13045|   Darrin Van Huff|Corporate|United States|  Los Angeles|California|      90036|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|  West|
|     6|CA-2015-115812|2015-06-09|2015-06-14|Standard Class|

26/03/08 09:38:27 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 956763 ms exceeds timeout 120000 ms
26/03/08 09:55:09 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/08 09:55:13 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o